# Week 1 · Train and diagnose a perceptron

Run this notebook with the project's Python 3.11 Conda kernel. It uses NumPy and Matplotlib with synthetic binary inputs. For a refresher on NAND and sigmoid training, try the optional `week-01_perceptron_warmup.ipynb` first. This lab adds the perceptron update, a boundary plot, and a controlled AND–XOR comparison.

By the end, you should be able to trace an update, compare scalar and batched predictions, and explain why a linear threshold depends on the representation.

Let $X\in\mathbb{R}^{n\times d}$ contain one example per row and $w\in\mathbb{R}^d$. The score vector $Xw+b$ has shape $(n,)$. Label scores strictly greater than zero as 1. At $w=(0,0)$ and $b=0$, the positive example $(1,1)$ has prediction 0. With $\eta=1$, the update adds $(1,1)$ to $w$ and adds 1 to $b$. This single update also changes scores for other points.

In [ ]:
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray

X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
and_targets = np.array([0, 0, 0, 1])
xor_targets = np.array([0, 1, 1, 0])


def predict(
    features: NDArray[np.float64], weights: NDArray[np.float64], bias: float
) -> NDArray[np.int_]:
    """Apply a strict binary threshold to one score per row."""
    return (features @ weights + bias > 0).astype(int)


def train(
    features: NDArray[np.float64],
    targets: NDArray[np.int_],
    order: tuple[int, ...],
    epochs: int = 12,
) -> tuple[NDArray[np.float64], float, list[int]]:
    """Train on one fixed row permutation per pass, recording post-pass errors.

    Expects a nonempty feature matrix, one binary label per row, and a row permutation.
    This lab uses zero initialization and a unit learning rate.
    """
    weights = np.zeros(features.shape[1])
    bias = 0.0
    errors = []
    for _ in range(epochs):
        for index in order:
            row = features[index]
            prediction = int(row @ weights + bias > 0)
            error = int(targets[index]) - prediction
            weights += error * row
            bias += error
        errors.append(int(np.count_nonzero(predict(features, weights, bias) != targets)))
    return weights, bias, errors


assert predict(X, np.zeros(2), 0.0).tolist() == [0, 0, 0, 0]
order = (0, 1, 2, 3)
and_weights, and_bias, and_errors = train(X, and_targets, order)
_, _, xor_errors = train(X, xor_targets, order)
print("AND errors after each pass:", and_errors)
print("XOR errors after each pass:", xor_errors)
assert and_errors[-1] == 0
assert all(error > 0 for error in xor_errors)

## Read the decision geometry

The dotted line marks $w_1 x_1+w_2 x_2+b=0$. Points on the line are assigned 0 under the strict threshold. A pass evaluates errors on all four rows after its updates, so the line plot compares models at a common point in training. The four rows exhaust the binary input domain and the curves describe training behavior.

In [ ]:
figure, (boundary, learning) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for label, color in [(0, "tab:blue"), (1, "tab:orange")]:
    points = X[and_targets == label]
    boundary.scatter(points[:, 0], points[:, 1], color=color, label=f"AND label {label}")
x1 = np.linspace(-0.25, 1.25, 100)
if and_weights[1] != 0:
    x2 = -(and_weights[0] * x1 + and_bias) / and_weights[1]
    boundary.plot(x1, x2, "k:", label="Score = 0")
elif and_weights[0] != 0:
    boundary.axvline(-and_bias / and_weights[0], linestyle=":", color="black", label="Score = 0")
boundary.set(xlabel="x₁", ylabel="x₂", xlim=(-0.25, 1.25), ylim=(-0.25, 1.25), title="AND boundary")
boundary.legend()

passes = range(1, len(and_errors) + 1)
learning.plot(passes, and_errors, "o-", label="AND")
learning.plot(passes, xor_errors, "s-", label="XOR")
learning.set(xlabel="Pass", ylabel="Errors out of four", title="Training error", yticks=range(5))
learning.legend()
plt.show()

## Guided experiment · Change the row order

Keep the feature matrix, initialization, learning rate, and pass budget fixed. For each row order, use that order throughout training for both gates. The metric is zero error by pass 12. Four rows have 24 possible orders, so the experiment enumerates all of them. There is no random sampling or test split here.

Before running, predict whether row order can change (a) the number of passes for AND and (b) whether raw XOR can be fit. Run the categorical comparison first, then inspect the pass counts.

In [ ]:
for gate, targets in [("AND", and_targets), ("XOR", xor_targets)]:
    first_perfect_pass = []
    for row_order in permutations(range(len(X))):
        _, _, errors = train(X, targets, row_order)
        first_pass = next((step for step, error in enumerate(errors, start=1) if error == 0), None)
        if first_pass is not None:
            first_perfect_pass.append(first_pass)
    print(f"{gate}: {len(first_perfect_pass)}/24 orders reached zero error")
    if first_perfect_pass:
        print(f"First perfect pass: {min(first_perfect_pass)} to {max(first_perfect_pass)}")

## Independent work · Gate classifier

1. Implement your own batch threshold function and validate that inputs have shape `(n, d)`, weights have shape `(d,)`, and values are finite. Check that a zero score returns label 0.
2. Find weights for AND and OR independently. Test all four inputs for each gate and repeat with the rows in a different order. Inspect one prediction by hand.
3. Derive why XOR has no solution under the raw two-feature representation. Record one measured result, the assumptions behind it, and what this experiment says about changing feature representations.

**Optional extension:** Add a product feature and re-test XOR. Check a compact baseline and state why the result depends on access to the interaction at decision time.

**Research connection:** An additive score can be a transparent baseline for routing analyst review. Before adding interactions, define what evidence would show improved review quality on a time-ordered holdout, and account for the cost of missed flags.

**Reference:** Michael Nielsen, [Neural Networks and Deep Learning, Chapter 1](http://neuralnetworksanddeeplearning.com/chap1.html), perceptrons and sigmoid neurons.